# FT Colab Bootstrap

Bootstrap pentru experimentele de fine-tuning, nu training final. Scopul primei rulari in cloud:

- confirma runtime GPU;
- cloneaza repo-ul in Colab;
- importa split-ul hard-fenced din `src/data_split.py`;
- verifica synthetic seeds impotriva `TEST_IDS`;
- construieste un JSONL de tip chat pentru SFT pe datele synthetic.

Training-ul real ramane o celula opt-in dupa ce alegem modelul open-source final.

In [ ]:
# Install runtime dependencies.
%pip -q install -U \
  "transformers>=4.45.0,<5" \
  "datasets" \
  "accelerate" \
  "peft" \
  "trl" \
  "bitsandbytes" \
  "sentencepiece" \
  "jsonschema" \
  "huggingface_hub"

In [ ]:
# Runtime + repo checkout.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SoftNestSol/Medical-Notes.git"
REPO_DIR = Path("/content/Medical-Notes")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "data_split.py").exists():
            os.chdir(candidate)
            break

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

print(f"IN_COLAB={IN_COLAB}")
print(f"ROOT={ROOT}")

In [ ]:
# GPU smoke check.
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    !nvidia-smi

In [ ]:
# Project imports + frozen split guard.
from data_split import POOL_IDS, TEST_IDS, assert_no_test_leakage
from SOTA_EVALUATION.json_schema import ANTECEDENTE_ENUM, LOCALIZARE_ENUM, NOTE_SCHEMA

print("TEST_IDS", len(TEST_IDS), sorted(TEST_IDS))
print("POOL_IDS", len(POOL_IDS), sorted(POOL_IDS))
assert_no_test_leakage(POOL_IDS, context="pool sanity check")

In [ ]:
# Paths + minimal readers.
import csv
import json
from jsonschema import Draft7Validator

SYNTH_ROOT = ROOT / "data" / "synthetic" / "GPT5.5"
SYNTH_TRANSCRIPTS = SYNTH_ROOT / "transcripts"
SYNTH_REFS = SYNTH_ROOT / "refs"
SEED_PAIRS = SYNTH_ROOT / "seed_pairs.tsv"
OUT_DIR = ROOT / "artifacts" / "ft_bootstrap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

schema_validator = Draft7Validator(NOTE_SCHEMA)

for path in [SYNTH_ROOT, SYNTH_TRANSCRIPTS, SYNTH_REFS, SEED_PAIRS]:
    if not path.exists():
        raise FileNotFoundError(path)

with SEED_PAIRS.open(encoding="utf-8", newline="") as f:
    seed_ids = [row["seed_id"] for row in csv.DictReader(f, delimiter="\t")]

assert_no_test_leakage(seed_ids, context="synthetic seed_pairs.tsv")
print(f"synthetic seeds OK: {seed_ids}")

In [ ]:
# Prompt used to format SFT examples.
SYSTEM_PROMPT = f"""Ești un sistem de extragere de informații clinice din transcrierile consultațiilor unui cabinet de chiropractică din România.

Primești o transcriere (dialog terapeut-pacient) și returnezi UN SINGUR obiect JSON cu informațiile extrase.

REGULA FUNDAMENTALĂ: dacă o informație NU este rostită explicit în conversație, câmpul rămâne gol (null sau listă goală). Nu deduce. Nu presupune. Nu completa pe baza intuiției clinice.

Schema exactă:
{{
  "motivul_prezentarii": <string sau null>,
  "evaluarea_durerii_vas": <int 0-10, listă de int 0-10 sau null>,
  "localizarea_durerii": <listă de slug-uri din enum>,
  "localizarea_durerii_alta": <string sau null>,
  "antecedente": <listă de slug-uri din enum>,
  "antecedente_altele": <string sau null>,
  "medicatie_actuala": <listă de obiecte {{"denumire": string, "doza": string sau null}}>,
  "evaluare_functionala_initiala": <string sau null>
}}

Slug-uri localizarea_durerii permise: {LOCALIZARE_ENUM}
Slug-uri antecedente permise: {ANTECEDENTE_ENUM}

Returnează DOAR JSON valid, fără markdown fences și fără text suplimentar. Folosește null, nu None. Listele goale sunt [].
"""

def make_sft_messages(transcript: str, note: dict) -> dict:
    errors = sorted(schema_validator.iter_errors(note), key=lambda e: e.path)
    if errors:
        first = errors[0]
        raise ValueError(f"schema error at {list(first.path)}: {first.message}")
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "--- TRANSCRIEREA CONSULTAȚIEI ---\n\n" + transcript.strip()},
            {"role": "assistant", "content": json.dumps(note, ensure_ascii=False, separators=(",", ":"))},
        ]
    }

In [ ]:
# Build synthetic chat JSONL for SFT condition 5/7.
synth_rows = []
missing_pairs = []

for transcript_path in sorted(SYNTH_TRANSCRIPTS.glob("synth_*.txt")):
    ref_path = SYNTH_REFS / f"{transcript_path.stem}.json"
    if not ref_path.exists():
        missing_pairs.append((transcript_path, ref_path))
        continue
    synth_rows.append(make_sft_messages(transcript_path.read_text(encoding="utf-8"), read_json(ref_path)))

if missing_pairs:
    raise FileNotFoundError(missing_pairs[:3])

synthetic_jsonl = OUT_DIR / "synthetic_chiro_sft_messages.jsonl"
write_jsonl(synthetic_jsonl, synth_rows)

print(f"synthetic SFT rows: {len(synth_rows)}")
print(f"wrote: {synthetic_jsonl}")
print(json.dumps(synth_rows[0], ensure_ascii=False, indent=2)[:1200])

In [ ]:
# MTS CSV probe. This is intentionally separate because validation CSV currently needs repair/robust parsing.
import pandas as pd

MTS_ROOT = ROOT / "data" / "mts_dialog_ro_augmented"
for csv_path in sorted(MTS_ROOT.glob("*_ro_augmented.csv")):
    try:
        df = pd.read_csv(csv_path)
        print(csv_path.name, df.shape, "OK")
    except Exception as exc:
        print(csv_path.name, "NEEDS_ATTENTION", type(exc).__name__, exc)

In [ ]:
# Optional Hugging Face login + tokenizer/model smoke test.
# Leave RUN_MODEL_LOAD=False until the GPU runtime is confirmed.
RUN_MODEL_LOAD = False
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

if RUN_MODEL_LOAD:
    from huggingface_hub import notebook_login
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    # notebook_login()  # uncomment for gated models such as Gemma/RoLlama if needed

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,
    )
    print("loaded", MODEL_ID)
else:
    print("Set RUN_MODEL_LOAD=True after the GPU runtime is connected.")